# Libraries

In [1]:
import psycopg2
import sqlalchemy as sa

import joblib
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer

# Removes the limit for the number of displayed columns
pd.set_option("display.max_columns", None)
# Sets the limit for the number of displayed rows
pd.set_option("display.max_rows", 200)    
# setting the precision of floating numbers to 4 decimal points
pd.set_option("display.float_format", lambda x: "%.4f" % x)

# Load data

In [3]:
# Load SQL data into a DataFrame
data = pd.read_sql(
    """
    SELECT *
    FROM staging.application_test
    """, engine)

data2 = pd.read_sql(
    """
    SELECT *
    FROM staging.bureau
    """, engine)

data3 = pd.read_sql(
    """
    SELECT *
    FROM staging.bureau_balance
    """, engine)

# Data preparation

In [4]:
df = data.copy()
bur_df = data2.copy()
bur_bal_df = data3.copy()

In [5]:
df.shape

(48744, 121)

In [6]:
bur_df.shape

(1716428, 17)

In [7]:
bur_bal_df.shape

(27299925, 3)

## Drop columns

In [8]:
df.drop([
        'cnt_children','name_type_suite','occupation_type','flag_mobil', 'flag_emp_phone','flag_work_phone',
        'flag_cont_mobile','flag_phone','flag_email','region_rating_client_w_city',
        'weekday_appr_process_start','hour_appr_process_start', 'reg_city_not_work_city','live_city_not_work_city',
        'reg_city_not_live_city','organization_type', 'basementarea_avg',
        'years_beginexpluatation_avg', 'years_build_avg', 'commonarea_avg', 'elevators_avg','entrances_avg', 
        'floorsmax_avg', 'floorsmin_avg', 'landarea_avg', 'livingapartments_avg','livingarea_avg',
        'nonlivingapartments_avg', 'nonlivingarea_avg', 'apartments_mode','basementarea_mode', 
        'years_beginexpluatation_mode', 'commonarea_mode','elevators_mode', 'entrances_mode',  
        'floorsmax_mode', 'floorsmin_mode', 'landarea_mode','livingapartments_mode', 'livingarea_mode', 
        'nonlivingapartments_mode', 'nonlivingarea_mode','apartments_medi', 'basementarea_medi', 
        'years_beginexpluatation_medi', 'years_build_medi','commonarea_medi', 'elevators_medi', 
        'entrances_medi', 'floorsmax_medi', 'floorsmin_medi','landarea_medi', 'livingapartments_medi', 
        'livingarea_medi', 'nonlivingapartments_medi', 'nonlivingarea_medi', 'fondkapremont_mode', 
        'housetype_mode', 'totalarea_mode','wallsmaterial_mode', 'emergencystate_mode', 
        'def_30_cnt_social_circle','obs_60_cnt_social_circle', 'def_60_cnt_social_circle', 
        'flag_document_2', 'flag_document_3', 'flag_document_4', 'flag_document_5','flag_document_6', 'flag_document_7', 
        'flag_document_8', 'flag_document_9', 'flag_document_10', 'flag_document_11', 'flag_document_12', 
        'flag_document_13','flag_document_14','flag_document_15', 'flag_document_16', 'flag_document_17', 
        'flag_document_18', 'flag_document_19','flag_document_20', 'flag_document_21','amt_req_credit_bureau_hour', 
        'amt_req_credit_bureau_day','amt_req_credit_bureau_week','amt_req_credit_bureau_mon', 'amt_req_credit_bureau_qrt', 
        ],axis=1,inplace=True)

## Map features

In [9]:
family_map = {
    1: 'Alone',
    2: 'Small', 3: 'Small', 4: 'Small', 
    5: 'Medium', 6: 'Medium', 
    7: 'Large', 8: 'Large', 9: 'Large', 10: 'Large', 11: 'Large', 
    12: 'Large', 13: 'Large', 14: 'Large', 15: 'Large', 16: 'Large', 20: 'Large'
}

employment_map = {
    'Pensioner': 'Retired',
    'Student': 'Unemployed', 'Unemployed': 'Unemployed',
    'Businessman': 'Employed','Commercial associate': 'Employed','Maternity leave': 'Employed',
    'State servant': 'Employed', 'Working': 'Employed'
}

education_map = {
    'Academic degree': 'Degrees',
    'Higher education': 'Higher education', 'Incomplete higher': 'Higher education',
    'Lower secondary': 'Secondary', 'Secondary / secondary special': 'Secondary',
}

marriage_map = {
    'Single / not married': 'Single',
    'Married': 'Married',
    'Civil marriage': 'Civil marriage',
    'Separated': 'Separated',
    'Widow': 'Widow',
    'Unknown': 'Unknown',
}

In [10]:
df['family_size_grouped'] = df['cnt_fam_members'].map(family_map)
df['employment_status'] = df['name_income_type'].map(employment_map)
df['education_level'] = df['name_education_type'].map(education_map)
df['marriage_status'] = df['name_family_status'].map(marriage_map)

In [11]:
df2 = bur_df.merge(bur_bal_df,on='sk_bureau_id', how='right')

In [12]:
status_map = {
    '0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5,
    'C': 0, 'X': 0,
}

In [13]:
df2['status'] = df2['status'].map(status_map)

## Transform feature

In [14]:
df['age'] = round(abs(df['days_birth']/365.25)).astype(int)
days_col = ['days_employed','days_registration','days_id_publish','own_car_age','days_last_phone_change']
df[days_col] = df[days_col].apply(lambda x : round(abs(x)))

In [15]:
df.drop(['days_birth','cnt_fam_members','name_income_type','name_education_type','name_family_status'],axis=1,inplace=True)

## Aggregate feature

### *bureau.csv*

In [16]:
bur_merge = pd.DataFrame()

bur_merge['sk_id_curr'] = pd.DataFrame(bur_df['sk_id_curr'].sort_values().unique())
bur_merge = pd.merge(bur_merge,bur_df.groupby('sk_id_curr').agg(total_loans=('sk_id_curr','count')).reset_index(), on='sk_id_curr',how='left')
bur_merge = pd.merge(bur_merge,bur_df[bur_df['credit_active'] == 'Active'].groupby('sk_id_curr').agg(active_count=('sk_id_curr','count')).reset_index(), on='sk_id_curr',how='left')
bur_merge = pd.merge(bur_merge,bur_df[bur_df['credit_active'] == 'Closed'].groupby('sk_id_curr').agg(closed_count=('sk_id_curr','count')).reset_index(), on='sk_id_curr',how='left')
bur_merge = pd.merge(bur_merge,bur_df[bur_df['credit_active'] == 'Bad debt'].groupby('sk_id_curr').agg(bad_debt_count=('sk_id_curr','count')).reset_index(), on='sk_id_curr',how='left')
bur_merge = pd.merge(bur_merge,bur_df[bur_df['credit_active'] == 'Sold'].groupby('sk_id_curr').agg(sold_credit_count=('sk_id_curr','count')).reset_index(), on='sk_id_curr',how='left')
bur_merge = pd.merge(bur_merge,bur_df[bur_df['credit_active'] == 'Closed'].groupby('sk_id_curr').agg(total_past_default=('amt_credit_sum_overdue','count')).reset_index(), on='sk_id_curr',how='left')

bur_merge = pd.merge(bur_merge,bur_df.groupby('sk_id_curr').agg(total_active_debt=('amt_credit_sum_debt','sum')).reset_index(), on='sk_id_curr',how='left')
bur_merge = pd.merge(bur_merge,bur_df.groupby('sk_id_curr').agg(total_active_credit=('amt_credit_sum','sum')).reset_index(), on='sk_id_curr',how='left')
bur_merge = pd.merge(bur_merge,bur_df.groupby('sk_id_curr').agg(total_active_overdue=('amt_credit_sum_overdue','sum')).reset_index(), on='sk_id_curr',how='left')

### *bureau_balance.csv*

In [17]:
bur_bal_merge = pd.DataFrame()

bur_bal_merge['sk_id_curr'] = pd.DataFrame(df2['sk_id_curr'].sort_values().unique())
bur_bal_merge = pd.merge(bur_bal_merge,df2.groupby('sk_id_curr').agg(max_deliq=('status','max')).reset_index(), on='sk_id_curr',how='left')
bur_bal_merge = pd.merge(bur_bal_merge,df2[df2['status'] > 0].groupby('sk_id_curr').agg(deliq_month=('status','sum')).reset_index(), on='sk_id_curr',how='left')
bur_bal_merge = pd.merge(bur_bal_merge,df2.groupby('sk_id_curr').agg(total_deliq_month=('sk_bureau_id','count')).reset_index(), on='sk_id_curr',how='left')

## Final dataframe

In [18]:
df_final = df.merge(bur_merge, on='sk_id_curr', how='left')

In [19]:
df_final = df_final.merge(bur_bal_merge, on='sk_id_curr',how='left')

In [20]:
df_final.shape

(48744, 44)

In [21]:
df_final.sort_values('sk_id_curr').head(20)

,sk_id_curr,name_contract_type,code_gender,flag_own_car,flag_own_realty,amt_income_total,amt_credit,amt_annuity,amt_goods_price,name_housing_type,region_population_relative,days_employed,days_registration,days_id_publish,own_car_age,region_rating_client,reg_region_not_live_region,reg_region_not_work_region,live_region_not_work_region,ext_source_1,ext_source_2,ext_source_3,apartments_avg,years_build_mode,obs_30_cnt_social_circle,days_last_phone_change,amt_req_credit_bureau_year,family_size_grouped,employment_status,education_level,marriage_status,age,total_loans,active_count,closed_count,bad_debt_count,sold_credit_count,total_past_default,total_active_debt,total_active_credit,total_active_overdue,max_deliq,deliq_month,total_deliq_month
0,100001,Cash loans,F,False,True,135000.0000,568800.0000,20560.5000,450000.0000,House / apartment,0.0188,2329,5170.0000,812,NaN,2,False,False,False,0.7526,0.7897,0.1595,0.0660,NaN,0.0000,1740.0000,0.0000,Small,Employed,Higher education,Married,53,7.0000,3.0000,4.0000,NaN,NaN,4.0000,596686.5000,1453365.0000,0.0000,1.0000,1.0000,172.0000
1,100005,Cash loans,M,False,True,99000.0000,222768.0000,17370.0000,180000.0000,House / apartment,0.0358,4469,9118.0000,1623,NaN,2,False,False,False,0.5650,0.2917,0.4330,NaN,NaN,0.0000,0.0000,3.0000,Small,Employed,Secondary,Married,49,3.0000,2.0000,1.0000,NaN,NaN,1.0000,568408.5000,657126.0000,0.0000,0.0000,NaN,21.0000
2,100013,Cash loans,M,True,True,202500.0000,663264.0000,69777.0000,630000.0000,House / apartment,0.0191,4458,2175.0000,3503,5.0000,2,False,False,False,NaN,0.6998,0.6110,NaN,NaN,0.0000,856.0000,4.0000,Small,Employed,Higher education,Married,55,4.0000,NaN,4.0000,NaN,NaN,4.0000,0.0000,2072280.0600,0.0000,1.0000,7.0000,230.0000
3,100028,Cash loans,F,False,True,315000.0000,1575000.0000,49018.5000,1575000.0000,House / apartment,0.0264,1866,2000.0000,4208,NaN,2,False,False,False,0.5257,0.5097,0.6127,0.3052,0.9608,0.0000,1805.0000,3.0000,Small,Employed,Secondary,Married,38,12.0000,5.0000,7.0000,NaN,NaN,7.0000,186304.5000,1520875.0800,0.0000,0.0000,NaN,560.0000
4,100038,Cash loans,M,True,False,180000.0000,625500.0000,32067.0000,625500.0000,House / apartment,0.0100,2191,4000.0000,4262,16.0000,2,False,False,False,0.2021,0.4257,NaN,NaN,NaN,0.0000,821.0000,NaN,Small,Employed,Secondary,Married,36,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,100042,Cash loans,F,True,True,270000.0000,959688.0000,34600.5000,810000.0000,House / apartment,0.0252,12009,6116.0000,2027,10.0000,2,False,False,False,NaN,0.6289,0.3928,0.2412,0.7648,0.0000,1705.0000,2.0000,Small,Employed,Secondary,Married,51,14.0000,8.0000,6.0000,NaN,NaN,6.0000,3074162.8950,9540657.0000,0.0000,2.0000,9.0000,615.0000
6,100057,Cash loans,M,True,True,180000.0000,499221.0000,22117.5000,373500.0000,House / apartment,0.0228,2580,10125.0000,241,3.0000,2,False,False,False,0.7609,0.5711,0.6513,NaN,NaN,1.0000,1182.0000,1.0000,Small,Employed,Higher education,Married,46,8.0000,1.0000,7.0000,NaN,NaN,7.0000,0.0000,2952260.1000,0.0000,0.0000,NaN,367.0000
7,100065,Cash loans,M,False,True,166500.0000,180000.0000,14220.0000,180000.0000,With parents,0.0051,1387,5063.0000,2055,NaN,2,False,False,False,0.5653,0.6130,0.3124,NaN,NaN,0.0000,1182.0000,2.0000,Alone,Employed,Higher education,Single,26,13.0000,4.0000,9.0000,NaN,NaN,9.0000,540273.5100,1710873.9450,0.0000,0.0000,NaN,310.0000
8,100066,Cash loans,F,False,True,315000.0000,364896.0000,28957.5000,315000.0000,House / apartment,0.0462,1013,1686.0000,3171,NaN,1,False,False,False,0.7185,0.8088,0.5227,0.1031,NaN,0.0000,829.0000,5.0000,Small,Employed,Higher education,Married,35,3.0000,1.0000,2.0000,NaN,NaN,2.0000,181687.5000,949500.0000,0.0000,1.0000,1.0000,56.0000
9,100067,Cash loans,F,True,True,162000.0000,45000.0000,5337.0000,45000.0000,House / apartment,0.0186,2625,8124.0000,3041,5.0000,2,False,False,False,0.2106,0.4448,0.1941,NaN,NaN,4.0000,1423.0000,2.0000,Small,Employed,Higher education,Civil marriage,28,17.0000,5.0000,12.0000,NaN,NaN,12.0000,1176970.5000,3237102.0000,0.000

# Data preprocessing

## Domain knowledge treatment

In [22]:
fix_columns = ['amt_req_credit_bureau_year','total_loans','active_count','closed_count','bad_debt_count','sold_credit_count','total_past_default',
               'total_active_debt', 'total_active_credit','total_active_overdue','max_deliq','deliq_month','total_deliq_month',
               'obs_30_cnt_social_circle']

In [23]:
df_final = df_final[df_final.code_gender != 'XNA']
df_final['total_active_debt'] = df_final['total_active_debt'].clip(lower=0)
df_final[fix_columns] = df_final[fix_columns].fillna(0)
df_final.dropna(axis=0,subset=('family_size_grouped','days_last_phone_change'),inplace=True)
df_final.loc[df_final['flag_own_car'] == False, 'own_car_age'] = 0
df_final['is_apartments_avg_missing'] = df_final['apartments_avg'].isna().astype(int)
df_final['is_years_build_missing'] = df_final['years_build_mode'].isna().astype(int)
df_final['ext_source_mean'] = df_final[['ext_source_1', 'ext_source_2', 'ext_source_3']].mean(axis=1)
df_final['ext_source_min']  = df_final[['ext_source_1', 'ext_source_2', 'ext_source_3']].min(axis=1)
df_final['days_employed'] = df_final['days_employed'].replace(365243, 0)

In [24]:
df_ids = df_final['sk_id_curr']
df_final = df_final.drop(columns=['sk_id_curr'])

In [25]:
df_final.shape

(48743, 47)

## Impute missing values

In [26]:
imputer = joblib.load('imputer.pkl')

In [27]:
df_final_num = df_final.describe(include=np.number).columns

In [28]:
df_final[df_final_num] = imputer.transform(df_final[df_final_num])

In [29]:
print(df_final.isna().sum())

name_contract_type             0
code_gender                    0
flag_own_car                   0
flag_own_realty                0
amt_income_total               0
amt_credit                     0
amt_annuity                    0
amt_goods_price                0
name_housing_type              0
region_population_relative     0
days_employed                  0
days_registration              0
days_id_publish                0
own_car_age                    0
region_rating_client           0
reg_region_not_live_region     0
reg_region_not_work_region     0
live_region_not_work_region    0
ext_source_1                   0
ext_source_2                   0
ext_source_3                   0
apartments_avg                 0
years_build_mode               0
obs_30_cnt_social_circle       0
days_last_phone_change         0
amt_req_credit_bureau_year     0
family_size_grouped            0
employment_status              0
education_level                0
marriage_status                0
age       

## Feature engineering

In [30]:
def add_financial_ratios(df_final):
    ## Ratios
    # Credit-Income ratio, higher -> higher risk, 2-5 medium
    df_final['credit_income_ratio'] = df_final['amt_credit'] / df_final['amt_income_total']
    
    # Annuity-Income ratio
    df_final['annuity_income_ratio'] = df_final['amt_annuity'] / df_final['amt_income_total']
    
    # Loan-Value ratio, if > 1 borrower borrowing extra, higher borrowing higher risk
    df_final['loan_to_value'] = df_final['amt_credit'] / df_final['amt_goods_price']
    
    # Employment-age ratio, higher safer
    df_final['employment_age_ratio'] = df_final['days_employed'] / df_final['age']
    
    # Credit-Utilization ratio, higher -> higher risk, 0.3-0.7 medium
    df_final['credit_utilization_ratio'] = df_final['total_active_debt'] / (df_final['total_active_credit'] + 1)
    
    # Debt-to-Income ratio
    df_final['debt_income_ratio'] = df_final['total_active_debt'] / df_final['amt_income_total']
    
    # Overdue-to-debt ratio
    df_final['overdue_debt_ratio'] = df_final['total_active_overdue'] / (df_final['total_active_debt'] + 1)
    
    # debt-to-credit ratio
    df_final['debt_credit_ratio'] = df_final['total_active_debt'] / (df_final['total_active_credit'] + 1)
    
    # loan exposure ratio
    df_final['loan_exposure_ratio'] = (df_final['total_active_debt'] + df_final['amt_credit']) / df_final['amt_income_total']
    
    # delinquenct ratio
    df_final['deliq_ratio'] = df_final['deliq_month'] / (df_final['total_deliq_month'] + 1)

    return df_final

In [31]:
df_final = add_financial_ratios(df_final)

In [32]:
df_final.shape

(48743, 57)

## Outlier treatment

In [33]:
cap_cols = [
    # Financial amounts
    'amt_income_total', 'amt_credit', 'amt_annuity', 'amt_goods_price',
    # Days columns
    'days_employed', 'days_registration', 'days_last_phone_change',   
    # age
    'own_car_age', 'obs_30_cnt_social_circle',   
    # counts
    'amt_req_credit_bureau_year', 'total_loans', 'active_count', 'closed_count', 'total_past_default', 'deliq_month', 'total_deliq_month', 
    # bureau aggregations
    'total_active_debt', 'total_active_credit', 'total_active_overdue',    
    # ratios
    'credit_income_ratio', 'annuity_income_ratio', 'loan_to_value', 'employment_age_ratio', 'credit_utilization_ratio',
    'debt_income_ratio', 'overdue_debt_ratio', 'debt_credit_ratio', 'loan_exposure_ratio', 'deliq_ratio',
] # 29

In [34]:
# Fit on train only
bounds = {col: df_final[col].quantile(0.99) for col in cap_cols}


# Apply to all
for col, upper in bounds.items():
       df_final[col] = df_final[col].clip(upper=upper)

In [35]:
df_final.shape

(48743, 57)

# Model fitting and submission

In [36]:
gs_xgb_pipeline = joblib.load('pipeline.pkl')

In [37]:
y_pred_proba = gs_xgb_pipeline.predict_proba(df_final)[:,1]

C:\Users\DELL\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [12, 13] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [38]:
submission = pd.DataFrame({
    'sk_id_curr': df_ids,
    'Target': y_pred_proba
})
submission.to_csv('submission.csv', index=False)
print(submission.head())

   sk_id_curr  Target
0      100001  0.2475
1      100005  0.6519
2      100013  0.1630
3      100028  0.0976
4      100038  0.6299
